# 01.5 — How you'll know it works

The last notebook classified twenty-two failures. Every one of them was found by
a number, not by reading answers.

This notebook is about why that distinction is the whole course.

## Try judging by eye

Two answers to the same question. Both fluent, both specific, both citing a real
document that genuinely exists in the corpus.

Decide which one is correct before scrolling.

In [1]:
question = 'How many working days of annual leave is a confirmed staff member entitled to?'

answer_a = '''Confirmed staff are entitled to 21 working days of paid annual leave
each calendar year, exclusive of public holidays. A maximum of 5 working days may
be carried into the following year.

Source: Sahel Microfinance Bank Employee Handbook, section 4.'''

answer_b = '''Confirmed staff are entitled to 25 working days of paid annual leave
each calendar year, exclusive of public holidays. A maximum of 10 working days may
be carried into the following year.

Source: Sahel Microfinance Bank Employee Handbook, section 4.'''

print(question)
print('\n--- A ---\n' + answer_a)
print('\n--- B ---\n' + answer_b)

How many working days of annual leave is a confirmed staff member entitled to?

--- A ---
Confirmed staff are entitled to 21 working days of paid annual leave
each calendar year, exclusive of public holidays. A maximum of 5 working days may
be carried into the following year.

Source: Sahel Microfinance Bank Employee Handbook, section 4.

--- B ---
Confirmed staff are entitled to 25 working days of paid annual leave
each calendar year, exclusive of public holidays. A maximum of 10 working days may
be carried into the following year.

Source: Sahel Microfinance Bank Employee Handbook, section 4.


Nothing in either answer tells you which is right.

Both quote a real document accurately. Both cite the same section. Neither
hallucinated anything. The difference is that **A quotes the 2023 edition and B
quotes the 2025 edition**, and only one of those is still policy.

You could only tell by going to the source. Now do that for thirty-six questions.
Then do it again after changing your chunk size. Then again after changing the
embedding model. Then again after adding a reranker.

That is why reading answers doesn't scale, and it's worse than not scaling — a
well-written wrong answer is *more* convincing than an awkward right one, so
judging by eye actively selects for fluency over correctness.

## What measurement needs

One thing: questions where you already know the answer.

That set has a name — a **golden set** — and building one is a real piece of work
that this course treats as a skill rather than a chore. Ours is in the repository.

In [2]:
import csv
from pathlib import Path
from collections import Counter

questions = list(csv.DictReader(
    open(Path('../../corpus/golden_questions.csv'), newline='', encoding='utf-8')))

print(f'{len(questions)} questions\n')
for q in questions[:2]:
    for field in ('id', 'question', 'expected_answer', 'source_document',
                  'difficulty', 'failure_class'):
        print(f'  {field:<17} {q[field]}')
    print()

38 questions

  id                Q01
  question          How many working days of annual leave is a confirmed staff member entitled to?
  expected_answer   25 working days
  source_document   sahel-employee-handbook-2025.pdf
  difficulty        hard
  failure_class     staleness

  id                Q02
  question          What is the domestic per diem for staff travelling on Bank business?
  expected_answer   NGN 60,000 per night
  source_document   sahel-employee-handbook-2025.pdf
  difficulty        hard
  failure_class     staleness



Six columns, and the last two are the ones people leave out.

`difficulty` and `failure_class` are what turn a score into a diagnosis. An
overall number tells you something is wrong. A number that's fine on easy
questions and collapses on one failure class tells you *what* is wrong, which is
the difference between a metric and a dashboard nobody looks at.

In [3]:
print('difficulty:')
for level, n in Counter(q['difficulty'] for q in questions).most_common():
    print(f'  {level:<8} {n}')

print('\nfailure classes:')
classes = Counter(c for q in questions for c in q['failure_class'].split('+'))
for cls, n in sorted(classes.items()):
    print(f'  {cls:<22} {n}')

difficulty:
  hard     23
  medium   11
  easy     4

failure classes:
  abstention             2
  amendment              3
  attribution            2
  chart-only             2
  clarification          1
  conflation             1
  email-thread           2
  embedded-newlines      1
  formula-no-cache       1
  hidden-sheet           1
  html-boilerplate       1
  lookup                 4
  multi-fact             1
  negation               5
  ocr-required           1
  page-spanning-table    1
  speaker-notes-only     2
  staleness              6
  table                  1
  table-band             2
  tracked-changes        1
  trailing-note          1
  trap                   1


Two of the questions have no source document at all. Those are deliberate: the
answer isn't in the corpus, and the correct behaviour is to say so. A system that
answers them is hallucinating, and you can only catch that if you planted the
trap on purpose.

## Then choose a metric — and check it against chance

This step gets skipped almost universally, and it's cheap.

A metric is only useful if a bad system scores badly on it. That isn't automatic.
Work out what **random guessing** would score before you trust any number.

In [4]:
import math

CHUNKS = 71          # the module 02 pipeline produces this many
PER_DOC = 10         # roughly, per document

for k in (1, 3, 5):
    p = 1 - math.comb(CHUNKS - PER_DOC, k) / math.comb(CHUNKS, k)
    print(f'random hit@{k}: {p:.2f}')

random hit@1: 0.14
random hit@3: 0.37
random hit@5: 0.54


A coin flip scores about **0.5 on hit@5** with this corpus.

Any metric where guessing gets half marks has almost no room left to tell you
anything — and in module 02 you'll watch that exact thing happen: the pipeline
scores 100% at hit@5 on every document it can read, which sounds excellent and
means nothing.

hit@1 leaves real distance between failure and success, so that's what module 02
uses. On a corpus of ten thousand chunks the calculation comes out differently
and hit@5 becomes reasonable. **Redo this arithmetic for your own corpus rather
than copying the choice.**

## What this doesn't give you

The scorer you'll build in module 02 is about twelve lines, and it is wrong in
four known ways. It's worth knowing them now so you don't over-trust the number.

**It scores documents, not chunks.** Retrieving the right document counts as a
success even when the retrieved chunk doesn't contain the answer. Two of our
questions score as hits despite being unanswerable, because their figures live
in a chart image.

**36 questions is too few.** A three-point move is roughly one question changing
its mind. You cannot distinguish that from noise, and you will want to.

**It ignores everything below rank 1.** A correct document at position 2 scores
the same as one that never appeared.

**One person wrote the questions**, in that person's vocabulary, about things
that person thought to ask. Real users are stranger than any of us.

None of that makes it useless. A flawed number you can watch move beats no
number, and it is honestly how this goes in practice — people hack a scorer
together, over-trust it, and find out later what it was hiding.

Module 06 is when you find out. It rebuilds this properly: chunk-level recall,
confidence intervals so signal separates from noise, and a question set built
with a process. Then it re-runs the conclusions you drew in modules 03, 04 and
05 — and some of them won't survive.

## End of module 01

Five things to carry forward:

1. RAG is: find the relevant text, put it in the prompt, ask. Everything else is
   engineering around those three steps.
2. Ask whether you need it. Small corpora fit in a prompt; structured data wants
   SQL; some jobs want fine-tuning or a platform you didn't build.
3. Two pipelines. Ingestion runs offline and is allowed to be slow. Querying runs
   while someone waits.
4. Most failures are ingestion failures. Twenty-one out of twenty-two, measured,
   on this corpus.
5. You cannot tell whether it works by reading the answers.

**Module 02** builds the pipeline and produces the number everything else is
measured against.